# Principal Component Analysis (PCA)

> **Core Idea:** Find new axes in the data that capture maximum variance, then project data onto fewer of those axes — reducing dimensions while preserving as much information as possible.

---

| Section | Topic |
|---------|-------|
| 1 | What is PCA & Why it Exists |
| 2 | Geometric Intuition |
| 3 | Why Variance is the Key |
| 4 | Covariance and Covariance Matrix |
| 5 | Eigendecomposition of the Covariance Matrix |
| 6 | Step-by-Step PCA Algorithm |
| 7 | PCA with Scikit-Learn |
| 8 | Explained Variance & Scree Plot |
| 9 | Finding the Optimal Number of Components |
| 10 | When PCA Does Not Work |
| 11 | Quick Reference |


## 1. What is PCA & Why Does It Exist?

### The Problem: Datasets Are High-Dimensional

Every column in your dataset is a dimension (feature). Real-world data grows very fast:

| Data Type | Dimensions |
|-----------|-----------|
| MNIST digit image (28×28 grayscale) | 784 |
| Color photo (256×256 RGB) | 196,608 |
| Text — bag-of-words representation | 3,000 – 50,000 |
| Gene expression data | 20,000+ |

This creates three compounding problems:

| Problem | Effect |
|---------|--------|
| **Slow training** | Every extra feature adds parameters and computation |
| **Overfitting** | Model learns noise in irrelevant dimensions |
| **No visualization** | You cannot plot data beyond 3D |

---

### The Photography Analogy

A photographer at a 3D football match projects the entire 3D scene onto a flat 2D sensor. Depth is partially lost, but the essential structure — shapes, positions, patterns — is preserved in the photo.

```
3D scene   ──[ camera ]──>  2D photograph   (some depth lost, structure preserved)
784-D MNIST ──[  PCA  ]──>  2D plot         (some variance lost, patterns preserved)
10-D data   ──[  PCA  ]──>  3D / 2D space
```

PCA is that camera. It finds the **best projection angle** — the one that preserves the most spread in the data.

---

### Benefits

| Benefit | Description |
|---------|-------------|
| **Faster algorithm execution** | Fewer features → fewer computations |
| **Visualization** | Compress 784-D or 10-D data to 2D/3D for plotting |
| **Noise reduction** | Low-variance dimensions often encode noise, not signal |
| **Decorrelation** | PCA components are always orthogonal (zero correlation) |

## 2. Geometric Intuition

### Example: Student Exam Scores

Suppose you have scores for 6 students across 3 subjects:

| Student | Math | Physics | Chemistry |
|---------|------|---------|-----------|
| A | 88 | 85 | 82 |
| B | 72 | 70 | 68 |
| C | 94 | 91 | 89 |
| D | 60 | 58 | 62 |
| E | 80 | 77 | 75 |
| F | 50 | 53 | 55 |

**Observation:** Math, Physics, and Chemistry scores move almost together — they're basically saying the same thing: *"how academically strong is this student?"*

> PCA detects this redundancy and compresses 3 correlated features into 1 principal component that captures the shared "academic ability" direction.

---

### PCA vs Feature Selection

- **Feature Selection:** Drop Physics and Chemistry, keep only Math → you lose info about the diagonal spread.
- **PCA (Feature Extraction):** Rotate the axes so the new axis *aligns with the direction of maximum spread* → nearly no information lost.

> PCA does not select existing columns. It **creates new axes** tilted to align with the natural directions of spread in the data.


## 3. Why Variance is the Key

**Variance** measures **spread** — how far data points are from their mean.

When you project data onto an axis, you want to **maximise the spread of the projections**. More spread = more distinguishable points = more information retained.

- If variance → 0, all projected points collapse to one location → useless.
- **Maximum variance = maximum information = best projection axis.**

PCA finds axes in this order:
- **PC1** = the axis that maximises variance of projected data.
- **PC2** = the next best orthogonal axis.
- **PC3, PC4, ...** = remaining orthogonal axes, each explaining less variance.

---

### 🎯 Interview Tip
> **Q: Why does PCA maximize variance?**  
> **A:** Because variance measures how much information a projection retains. Maximizing variance means projected points are as spread out (distinguishable) as possible. A zero-variance projection is useless — all points collapse to a single value.


## 4. Covariance and Covariance Matrix

**Variance** measures the spread of a single variable. **Covariance** measures how two variables move together.

| Covariance sign | Meaning |
|---|---------|
| Positive | Both variables increase together |
| Negative | One increases while the other decreases |
| Zero | No linear relationship |

For a dataset with $d$ features, the **covariance matrix** $\Sigma$ is a $d \times d$ symmetric matrix:
- **Diagonal entries** — variance of each individual feature
- **Off-diagonal entries** — covariance between each pair of features

This matrix is the central object in PCA. It captures all the pairwise relationships between features, and PCA is essentially finding the most informative directions within it.


## 5. Eigendecomposition of the Covariance Matrix

To find the directions of maximum variance, PCA solves the equation:

$$\Sigma \vec{u} = \lambda \vec{u}$$

This is the **eigenvalue equation**. The solution is:
- **Eigenvectors** $\vec{u}$ — the directions (principal component axes)
- **Eigenvalues** $\lambda$ — the amount of variance captured along each direction

| Concept | Role in PCA |
|---|---|
| Eigenvector with the largest eigenvalue | PC1 — direction of maximum variance |
| Eigenvector with 2nd largest eigenvalue | PC2 — orthogonal to PC1 |
| All eigenvectors sorted by eigenvalue | Ranked principal components |

The fraction of total variance explained by component $k$:

$$\text{Variance explained by PC}_k = \frac{\lambda_k}{\sum_{j} \lambda_j}$$

Since the covariance matrix is symmetric, its eigenvectors are always **orthogonal** — meaning all PCA components are uncorrelated with each other.

---

**Why does this work?**  
Each eigenvector points along a natural axis of the data's spread. The eigenvalue tells you how much spread (variance) exists along that axis. By picking the top-k eigenvectors, you keep the directions that contain the most information.


## 6. Step-by-Step PCA Algorithm

Given a dataset $X$ of shape $(n \times d)$, to reduce it to $k$ dimensions:

**Step 1 — Mean Center the Data**  
Subtract the column mean from each feature. This shifts the origin to the center of the data cloud so that the covariance matrix correctly measures spread around the mean, not around zero.

**Step 2 — Compute the Covariance Matrix**  
Build the $d \times d$ covariance matrix from the centered data.

**Step 3 — Eigendecomposition**  
Compute all eigenvalues and eigenvectors of the covariance matrix. Sort them in descending order of eigenvalue.

**Step 4 — Select Top-k Eigenvectors**  
Pick the $k$ eigenvectors with the largest eigenvalues. Stack them into a projection matrix $W$ of shape $(d \times k)$.

**Step 5 — Project the Data**  
Multiply the centered data by $W$. Each row in the result is the original sample represented in the new $k$-dimensional space.

```
Original data (n × d)
        ↓  Mean center
X_centered (n × d)
        ↓  Covariance matrix
Σ (d × d)
        ↓  Eigendecomposition → sort descending
λ₁ ≥ λ₂ ≥ ... ≥ λd   with eigenvectors v₁, v₂, ..., vd
        ↓  Select top-k → W (d × k)
        ↓  Project
X_reduced (n × k)
```


## 7. PCA with Scikit-Learn

Sklearn's `PCA` uses **SVD (Singular Value Decomposition)** internally rather than explicit eigendecomposition. It is numerically more stable and produces identical results.

### Key API

```python
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)      # always scale before PCA

pca = PCA(n_components=k)               # fixed number of components
# or
pca = PCA(n_components=0.95)            # keep enough components to retain 95% variance

X_reduced = pca.fit_transform(X_scaled)
```

### Important Attributes

| Attribute | Description |
|-----------|-------------|
| `pca.components_` | Shape $(k \times d)$ — the principal axes (eigenvectors) |
| `pca.explained_variance_` | Variance captured per component (eigenvalues) |
| `pca.explained_variance_ratio_` | Fraction of total variance per component |
| `pca.mean_` | Per-feature mean used for centering |
| `pca.n_components_` | Actual number of components retained |


In [1]:
import numpy as np
from sklearn.datasets import load_iris

data = load_iris()
X = data.data
y = data.target

print(f"Dataset: Iris  |  Shape: {X.shape}  |  Features: {list(data.feature_names)}")


Dataset: Iris  |  Shape: (150, 4)  |  Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize first (critical step!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Sklearn PCA
pca_sklearn = PCA(n_components=2)
X_pca = pca_sklearn.fit_transform(X_scaled)

print("=== Sklearn PCA Results ===")
print(f"Reduced shape:            {X_pca.shape}")
print(f"Explained variance ratio: {pca_sklearn.explained_variance_ratio_.round(4)}")
print(f"Total variance explained: {pca_sklearn.explained_variance_ratio_.sum():.4f}")
print(f"\nPrincipal components (eigenvectors):\n{pca_sklearn.components_.round(4)}")


=== Sklearn PCA Results ===
Reduced shape:            (150, 2)
Explained variance ratio: [0.7296 0.2285]
Total variance explained: 0.9581

Principal components (eigenvectors):
[[ 0.5211 -0.2693  0.5804  0.5649]
 [ 0.3774  0.9233  0.0245  0.0669]]


## 8. Explained Variance & Scree Plot

After fitting PCA on all components, `explained_variance_ratio_` tells you what fraction of the total variance each component captures. The cumulative sum tells you how much information is retained if you stop at $k$ components.

$$\text{Explained Variance Ratio}_k = \frac{\lambda_k}{\sum_{j=1}^{d} \lambda_j}$$

### The Scree Plot

A scree plot graphs individual explained variance (eigenvalues) against the component number. It helps you visually identify the "elbow point" where adding more components yields diminishing returns in explained variance.

## 9. Finding the Optimal Number of Components

| Strategy | How | When to Use |
|----------|-----|-------------|
| **Variance threshold** | Keep components until cumulative variance reaches 95% or 90% | Most common default |
| **Elbow method** | Find the bend in the scree plot | When there is no fixed threshold requirement |
| **Cross-validation** | Tune `n_components` as a hyperparameter | When optimizing downstream model performance |

**Variance threshold — Sklearn shortcut:**

```python
pca = PCA(n_components=0.95)   # retains the minimum number of components needed for 95% variance
```

**Elbow rule:** look at the scree plot and find where the curve stops dropping steeply. Before the elbow, each component adds significant information. After it, gains are marginal.


In [3]:
# Strategy 1: Auto-select components to preserve 95% variance

pca_95 = PCA(n_components=0.95)
X_95 = pca_95.fit_transform(X_scaled)
print(f"Components to preserve 95% variance: {pca_95.n_components_}")
print(f"Achieved variance: {pca_95.explained_variance_ratio_.sum()*100:.2f}%")

# Strategy 2: Cross-validation to find optimal k
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

scores = {}
for k in range(1, X_scaled.shape[1] + 1):
    pipe = Pipeline([
        ('pca',  PCA(n_components=k)),
        ('clf',  LogisticRegression(max_iter=500, random_state=42))
    ])
    cv_score = cross_val_score(pipe, X_scaled, y, cv=5, scoring='accuracy').mean()
    scores[k] = cv_score

print("\nCV Accuracy by number of PCA components:")
for k, score in scores.items():
    bar = '█' * int(score * 50)
    print(f"  k={k}: {score:.4f}  {bar}")

best_k = max(scores, key=scores.get)
print(f"\nBest k = {best_k}  (CV accuracy = {scores[best_k]:.4f})")


Components to preserve 95% variance: 2
Achieved variance: 95.81%

CV Accuracy by number of PCA components:
  k=1: 0.9267  ██████████████████████████████████████████████
  k=2: 0.9133  █████████████████████████████████████████████
  k=3: 0.9600  ████████████████████████████████████████████████
  k=4: 0.9600  ████████████████████████████████████████████████

Best k = 3  (CV accuracy = 0.9600)


## 10. When PCA Does Not Work

| Failure Mode | Why PCA Fails | Alternative |
|---|---|---|
| Non-linear structure | PCA only finds linear directions; curved manifolds are missed | Kernel PCA, t-SNE, UMAP |
| Class-discriminative reduction | PCA ignores labels and may discard class-separating dimensions | LDA (Linear Discriminant Analysis) |
| Sparse or high-dimensional text | Expensive on 50,000-D bag-of-words vectors | TruncatedSVD (LSA) |
| Non-Gaussian distributions | PCA assumes variance along linear subspaces | ICA (Independent Component Analysis) |
| Unscaled features | Features with large numeric ranges dominate the variance | Apply StandardScaler before PCA |


### Case 1: Circular / Spherical Data (Equal Variance in All Directions)

When data is distributed uniformly in a circle (2D) or a sphere (higher dimensions), the variance is **equal in every direction**. Think of a ball of data points — no matter which angle you look at it from, the spread is the same. PCA works by finding the direction with the **most** spread, but here every direction has the same spread. All eigenvalues come out nearly equal, so no single principal component stands out as more important than others.

If you reduce from 2D to 1D, you lose roughly 50% of the variance **no matter which axis you pick**, because there is no "best angle." PCA has nothing meaningful to compress.

> **Key idea:** PCA needs directions of **unequal variance** to be useful. If the data cloud is spherical or isotropic, dimensionality reduction will always destroy a significant portion of the structure.

---

### Case 2: Non-Linear Relationships (e.g., $y = x^2$ — Parabolic Data)

PCA can only discover **straight-line** directions. When the true relationship in the data is **curved** — like a parabola ($y = x^2$) — PCA fails badly.

Imagine data shaped like an upside-down U (a parabola). The horizontal direction has the widest spread, so PCA picks it as PC1. But projecting onto that horizontal axis **collapses both arms of the parabola on top of each other**. Points that were far apart along the curve now land at the same location. The information about which arm a point belongs to is completely destroyed — even though PCA technically captured the "maximum variance" direction.

No single straight axis can faithfully represent a curved structure. The data lives on a **non-linear manifold**, and PCA's linear projection flattens it in a lossy way.

> **Key idea:** If the important structure in your data is **curved or non-linear**, PCA will mix together points that should remain separate. Use **Kernel PCA**, **t-SNE**, or **UMAP** instead.

---

### When to Suspect PCA Will Fail

| Symptom | Likely Cause | Better Alternative |
|---|---|---|
| All components explain roughly equal variance | Spherical / isotropic data | Data may not be compressible; try non-linear methods |
| Downstream model accuracy drops sharply after PCA | PCA discarded class-separating dimensions | LDA (supervised reduction) |
| 2D PCA plot shows overlapping clusters that you know are separable | Non-linear boundaries between classes | Kernel PCA, t-SNE, UMAP |
| Scree plot has no clear elbow | Variance is spread evenly — no low-rank structure | Consider if reduction is needed at all |

## Advantages and Disadvantages of PCA

### Advantages

| Advantage | Explanation |
|-----------|-------------|
| **Dimensionality reduction** | Compresses hundreds or thousands of features into a handful of components, making downstream algorithms faster and cheaper to train |
| **Noise removal** | Low-variance components often capture sensor noise or irrelevant fluctuations. Dropping them acts as a built-in denoiser |
| **Eliminates multicollinearity** | Principal components are orthogonal (zero correlation), which removes the correlated-feature problem that hurts models like linear regression |
| **Visualization** | Projects high-dimensional data into 2D or 3D so you can visually inspect clusters, outliers, and class separability |
| **Unsupervised — no labels needed** | Works purely on the feature matrix, so it can be applied even when you have no target variable |
| **Improves generalization** | By reducing the feature space, PCA lowers the risk of overfitting, especially when the number of features is large relative to the number of samples |

### Disadvantages

| Disadvantage | Explanation |
|--------------|-------------|
| **Loss of interpretability** | Each principal component is a weighted mix of all original features. You can no longer say "this feature matters" — the components are abstract directions |
| **Information loss** | Dropping lower-variance components always discards some information. If the discarded variance was meaningful, model performance can degrade |
| **Linearity assumption** | PCA only captures linear relationships. Curved or non-linear structures (e.g., spirals, parabolas) are not faithfully represented |
| **Label-blind** | PCA maximizes total variance, not class separability. It may throw away exactly the dimensions that distinguish one class from another |
| **Scale sensitivity** | Features with larger numeric ranges dominate the covariance matrix. You **must** standardize before applying PCA, or the results are meaningless |
| **Computational cost on very high dimensions** | Computing the full covariance matrix is $O(d^2 \cdot n)$. For extremely high-dimensional sparse data (e.g., 50,000-D text), this becomes expensive — use TruncatedSVD instead |

## 11. Quick Reference

### Interview Questions

**What is PCA?**  
PCA finds new axes aligned with the directions of greatest variance in the data. Projecting onto a smaller number of these axes reduces dimensions while retaining as much information as possible.

**Why standardize before PCA?**  
PCA is driven by variance. A feature with large values will dominate just because of its scale. Standardizing ensures every feature contributes equally.

**What is a principal component?**  
A principal component is a linear combination of all original features. It corresponds to an eigenvector of the covariance matrix. PC1 captures the most variance, PC2 captures the next most while being orthogonal to PC1, and so on.

**PCA vs Feature Selection — what is the difference?**  
Feature selection keeps a subset of existing columns. PCA creates entirely new columns as combinations of all originals. PCA retains more information for the same number of dimensions, but the resulting components are harder to interpret.

**What are the main limitations of PCA?**  
- Only captures linear structure
- Ignores class labels — may remove exactly the dimensions that separate classes
- Sensitive to feature scale — standardization is required
- Components are abstract and difficult to interpret

---

### Cheatsheet

| Concept | Key Point |
|---------|-----------|
| Covariance matrix | $d \times d$ symmetric matrix encoding pairwise feature relationships |
| Principal component | Eigenvector of the covariance matrix |
| Variance explained by PC$_k$ | $\lambda_k \;/\; \sum_j \lambda_j$ |
| Reconstruction | $\hat{X} = X_{\text{reduced}} \cdot W^T + \bar{X}$ |
| Auto variance threshold | `PCA(n_components=0.95)` |

---

### Alternatives to PCA

| Method | Use Case |
|--------|----------|
| LDA | Supervised dimensionality reduction |
| Kernel PCA | Non-linear structure |
| t-SNE / UMAP | 2D/3D visualization |
| TruncatedSVD | Sparse matrices (text/NLP) |
| Autoencoders | Deep, non-linear reduction |
